# Ablation B: Replay Ratio 5%
Runs a single `run_experiment` with `replay_ratio=0.05` (EWC + Replay).

**GPU required.**

### Google Drive folder structure expected:
```
MyDrive/pi-detection/
  utils/utils.py
  data/t1_llmail.parquet
  data/t2_hackaprompt.parquet
  data/t3_bipia.parquet
  results/
  checkpoints/
```

### Outputs
- `results/results_replay_ratio_5pct.json` saved directly to Drive


## 1. Install & Setup

In [ ]:
!pip install -q transformers accelerate scikit-learn

from google.colab import drive
drive.mount('/content/drive')

import os, sys, gc
import torch

BASE_DIR   = "/content/drive/MyDrive/CMPE 401/Antidote"
UTILS_DIR  = os.path.join(BASE_DIR, 'utils')
DATA_DIR   = os.path.join(BASE_DIR, 'pi-detection-data')
RESULTS_DIR = os.path.join(BASE_DIR, '5_pct/results')
CKPT_DIR   = os.path.join(BASE_DIR, '5_pct/checkpoints')

sys.path.append(UTILS_DIR)
from utils import *

# Override CFG paths to point to Drive
CFG['checkpoint_dir'] = CKPT_DIR
CFG['results_dir']    = RESULTS_DIR
CFG['data_dir']       = DATA_DIR

os.makedirs(CKPT_DIR,    exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Mounted at /content/drive
Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


## 2. HuggingFace Login

In [2]:
from google.colab import userdata
from huggingface_hub import login

login(userdata.get('HF_TOKEN'))

## 3. Load Data

In [3]:
import pandas as pd
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])
tasks     = {}

for task_name, fname in [
    ('T1_LLMail',      't1_llmail.parquet'),
    ('T2_HackAPrompt', 't2_hackaprompt.parquet'),
    ('T3_BIPIA',       't3_bipia.parquet'),
]:
    path = os.path.join(DATA_DIR, fname)
    if os.path.exists(path):
        df = pd.read_parquet(path)
        tr, va, te = make_loaders(df, tokenizer)
        tasks[task_name] = {'train': tr, 'val': va, 'test': te, 'df': df}
        print(f'  {task_name}: loaded')
    else:
        print(f'  {task_name}: NOT FOUND — check your Drive path')

print(f'Loaded {len(tasks)} tasks.')

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

  T1_LLMail: loaded
  T2_HackAPrompt: loaded
  T3_BIPIA: loaded
Loaded 3 tasks.


## 4. OOM Patches

In [4]:
import gc, os
import utils as _utils

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# Patch 1: flush GPU after checkpoint save
original_save = _utils.save_checkpoint

def save_checkpoint_and_flush(model, path):
    original_save(model, path)
    gc.collect()
    torch.cuda.empty_cache()
    free = (torch.cuda.get_device_properties(0).total_memory
            - torch.cuda.memory_allocated()) / 1e9
    print(f'  GPU cache cleared. Free VRAM: {free:.2f}GB')

_utils.save_checkpoint = save_checkpoint_and_flush

# Patch 2: gradient checkpointing to reduce activation memory
original_load_model = _utils.load_model

def load_model_with_gc():
    model = original_load_model()
    model.gradient_checkpointing_enable()
    print('  Gradient checkpointing enabled.')
    return model

_utils.load_model = load_model_with_gc

# Patch 3: clear GPU between tasks (correct scoping)
original_run_experiment = _utils.run_experiment
original_tt = _utils.train_task

def run_experiment_with_cleanup(experiment_name, tasks, tokenizer,
                                use_ewc=False, use_replay=False, joint_training=False):
    def train_task_patched(*args, **kwargs):
        result = original_tt(*args, **kwargs)
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        free = (torch.cuda.get_device_properties(0).total_memory
                - torch.cuda.memory_allocated()) / 1e9
        print(f'  Inter-task GPU cleared. Free VRAM: {free:.2f}GB')
        return result

    _utils.train_task = train_task_patched
    try:
        result = original_run_experiment(
            experiment_name, tasks, tokenizer,
            use_ewc=use_ewc, use_replay=use_replay, joint_training=joint_training
        )
    finally:
        _utils.train_task = original_tt

    return result

# Patch 4: move EWC Fisher matrices to CPU after registration to free VRAM
original_register_task = _utils.EWC.register_task

def register_task_cpu_offload(self, model, data_loader, task_name):
    original_register_task(self, model, data_loader, task_name)
    for task in self.fisher:
        for n in self.fisher[task]:
            self.fisher[task][n] = self.fisher[task][n].cpu()
    for task in self.params:
        for n in self.params[task]:
            self.params[task][n] = self.params[task][n].cpu()
    gc.collect()
    torch.cuda.empty_cache()
    free = (torch.cuda.get_device_properties(0).total_memory
            - torch.cuda.memory_allocated()) / 1e9
    print(f'  EWC data offloaded to CPU. Free VRAM: {free:.2f}GB')

_utils.EWC.register_task = register_task_cpu_offload
_utils.run_experiment = run_experiment_with_cleanup
run_experiment = run_experiment_with_cleanup

print('All patches active (with gradient checkpointing).')

All patches active (with gradient checkpointing).


## 5. Run Ablation B1: Replay Ratio 5%

In [5]:
CFG['epochs_per_task'] = 2
CFG['replay_ratio'] = 0.05

key    = 'replay_ratio_5pct'
result = {}

result[key], _model = run_experiment(
    key, tasks, tokenizer,
    use_ewc=True, use_replay=True, joint_training=False,
)
del _model
gc.collect()
torch.cuda.empty_cache()

# Save directly to Drive so results persist even if session disconnects
save_results(result, os.path.join(RESULTS_DIR, "results_replay_ratio_5pct.json"))
print(f'\nDone. Saved: results_replay_ratio_5pct.json')

`torch_dtype` is deprecated! Use `dtype` instead!



EXPERIMENT: replay_ratio_5pct
  EWC: True | Replay: True | Joint: False


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight        

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

  Gradient checkpointing enabled.

--- Training on T1_LLMail (step 1/3) ---
  Epoch 1/2 | Train Loss: 0.1450 | Val F1: 0.9845 | Val Loss: 0.0655
  Epoch 2/2 | Train Loss: 0.0443 | Val F1: 0.9898 | Val Loss: 0.0478
  Best val F1: 0.9898
  Inter-task GPU cleared. Free VRAM: 14.14GB
  Computing Fisher matrix for T1_LLMail...
  EWC registered T1_LLMail. Total tasks: 1
  EWC data offloaded to CPU. Free VRAM: 14.13GB
  Replay buffer: +5000 from T1_LLMail. Total: 5000
  After T1_LLMail → T1_LLMail test F1: 0.9933
  Checkpoint saved: /content/drive/MyDrive/CMPE 401/Antidote/5_pct/checkpoints/replay_ratio_5pct_after_T1_LLMail.pt
  GPU cache cleared. Free VRAM: 14.13GB

--- Training on T2_HackAPrompt (step 2/3) ---
  Epoch 1/2 | Train Loss: 0.7570 | Val F1: 0.6633 | Val Loss: 0.5913
  Epoch 2/2 | Train Loss: 0.5909 | Val F1: 0.7099 | Val Loss: 0.5674
  Best val F1: 0.7099
  Inter-task GPU cleared. Free VRAM: 14.14GB
  Computing Fisher matrix for T2_HackAPrompt...
  EWC registered T2_HackAPrompt.